In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Libraries imported successfully")

# %% [markdown]
# ### 2. Load Raw Data

# %%
# Load the dataset
# Note: Using October 2019 data - adjust filename if using different month
file_path = '../data/raw/2019-Oct.csv'

print("Loading data...")
df = pd.read_csv(file_path)

print(f"✅ Data loaded successfully")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

# %% [markdown]
# ### 3. Initial Data Exploration

# %%
# Basic information
print("Dataset Information:")
print("=" * 50)
df.info()

print("\n\nBasic Statistics:")
print("=" * 50)
df.describe()

print("\n\nMissing Values:")
print("=" * 50)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])

# %%
# Check unique values
print("Unique Value Counts:")
print("=" * 50)
for col in df.columns:
    print(f"{col}: {df[col].nunique():,} unique values")

# %%
# Event type distribution
print("\nEvent Type Distribution:")
print("=" * 50)
event_dist = df['event_type'].value_counts()
print(event_dist)
print(f"\nPercentages:")
print((event_dist / len(df) * 100).round(2))

# %% [markdown]
# ### 4. Data Cleaning

# %%
# Create a copy for cleaning
df_clean = df.copy()

print("Initial shape:", df_clean.shape)

# Step 1: Convert timestamp
print("\n1. Converting event_time to datetime...")
df_clean['event_time'] = pd.to_datetime(df_clean['event_time'])
print("✅ Timestamp converted")

# Step 2: Handle missing values
print("\n2. Handling missing values...")

# Drop rows with missing essential columns
essential_cols = ['event_time', 'event_type', 'user_id']
df_clean = df_clean.dropna(subset=essential_cols)
print(f"✅ Dropped rows with missing essential data. New shape: {df_clean.shape}")

# Fill missing brand with 'Unknown'
df_clean['brand'] = df_clean['brand'].fillna('Unknown')

# Fill missing category_code with 'Uncategorized'
df_clean['category_code'] = df_clean['category_code'].fillna('Uncategorized')

# Fill missing price with median price
median_price = df_clean['price'].median()
df_clean['price'] = df_clean['price'].fillna(median_price)

print("✅ Missing values handled")

# Step 3: Remove duplicates
print("\n3. Removing duplicates...")
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates()
after_dedup = len(df_clean)
print(f"✅ Removed {before_dedup - after_dedup:,} duplicate rows")

# Step 4: Sort by time
print("\n4. Sorting by time...")
df_clean = df_clean.sort_values('event_time').reset_index(drop=True)
print("✅ Data sorted chronologically")

# Step 5: Remove outliers in price
print("\n5. Handling price outliers...")
Q1 = df_clean['price'].quantile(0.01)
Q3 = df_clean['price'].quantile(0.99)
df_clean = df_clean[(df_clean['price'] >= Q1) & (df_clean['price'] <= Q3)]
print(f"✅ Filtered to reasonable price range. New shape: {df_clean.shape}")

print("\n" + "=" * 50)
print(f"FINAL CLEANED SHAPE: {df_clean.shape}")
print("=" * 50)

# %% [markdown]
# ### 5. Feature Engineering

# %%
print("Creating temporal features...")

# Extract temporal components
df_clean['date'] = df_clean['event_time'].dt.date
df_clean['hour'] = df_clean['event_time'].dt.hour
df_clean['day_of_week'] = df_clean['event_time'].dt.dayofweek
df_clean['day_name'] = df_clean['event_time'].dt.day_name()
df_clean['day_of_month'] = df_clean['event_time'].dt.day
df_clean['week'] = df_clean['event_time'].dt.isocalendar().week
df_clean['month'] = df_clean['event_time'].dt.month
df_clean['year'] = df_clean['event_time'].dt.year

# Derived features
df_clean['is_weekend'] = df_clean['day_of_week'].isin([5, 6]).astype(int)
df_clean['is_business_hours'] = df_clean['hour'].between(9, 17).astype(int)

# Time of day categorization
def categorize_time(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    elif 18 <= hour < 24:
        return 'Evening'
    else:
        return 'Night'

df_clean['time_of_day'] = df_clean['hour'].apply(categorize_time)

# Price categories
df_clean['price_category'] = pd.cut(
    df_clean['price'],
    bins=[0, 10, 50, 100, 500, float('inf')],
    labels=['Budget', 'Low', 'Medium', 'High', 'Premium']
)

print("✅ Temporal features created")
print(f"\nNew columns: {df_clean.columns.tolist()[-15:]}")

# %%
# Category hierarchy extraction
print("\nExtracting category hierarchy...")

# Split category_code into main and sub categories
df_clean['main_category'] = df_clean['category_code'].str.split('.').str[0]
df_clean['sub_category'] = df_clean['category_code'].str.split('.').str[1]

print("✅ Category hierarchy extracted")

# %% [markdown]
# ### 6. Create Aggregated Views

# %%
print("Creating aggregated temporal views...")

# Hourly aggregation
hourly = df_clean.groupby(df_clean['event_time'].dt.floor('H')).agg({
    'user_id': 'nunique',
    'event_type': 'count',
    'product_id': 'nunique',
    'price': 'sum',
    'user_session': 'nunique'
}).reset_index()

hourly.columns = ['event_time', 'unique_users', 'total_events', 'unique_products', 'total_revenue', 'unique_sessions']

print(f"✅ Hourly aggregation: {hourly.shape}")

# %%
# Daily aggregation
daily = df_clean.groupby('date').agg({
    'user_id': 'nunique',
    'event_type': 'count',
    'product_id': 'nunique',
    'price': 'sum',
    'user_session': 'nunique'
}).reset_index()

daily.columns = ['date', 'unique_users', 'total_events', 'unique_products', 'total_revenue', 'unique_sessions']
daily['date'] = pd.to_datetime(daily['date'])

print(f"✅ Daily aggregation: {daily.shape}")

# %%
# Event-specific aggregations
event_summary = df_clean.groupby(['date', 'event_type']).size().reset_index(name='count')
event_pivot = event_summary.pivot(index='date', columns='event_type', values='count').fillna(0)
event_pivot = event_pivot.reset_index()
event_pivot['date'] = pd.to_datetime(event_pivot['date'])

print(f"✅ Event-specific aggregation: {event_pivot.shape}")

# %% [markdown]
# ### 7. Data Quality Check

# %%
print("Data Quality Report:")
print("=" * 50)

# Time range
print(f"Time Range: {df_clean['event_time'].min()} to {df_clean['event_time'].max()}")
print(f"Duration: {(df_clean['event_time'].max() - df_clean['event_time'].min()).days} days")

# Event counts
print(f"\nTotal Events: {len(df_clean):,}")
print(f"Unique Users: {df_clean['user_id'].nunique():,}")
print(f"Unique Products: {df_clean['product_id'].nunique():,}")
print(f"Unique Sessions: {df_clean['user_session'].nunique():,}")

# Event distribution
print(f"\nEvent Distribution:")
print(df_clean['event_type'].value_counts())

# Data completeness
print(f"\nData Completeness:")
completeness = (1 - df_clean.isnull().sum() / len(df_clean)) * 100
print(completeness.round(2))

print("\n✅ Data quality check complete")

# %% [markdown]
# ### 8. Save Processed Data

# %%
print("Saving processed datasets...")

# Save main cleaned dataset (sample if too large)
if len(df_clean) > 1000000:
    # Save full dataset
    df_clean.to_csv('../data/processed/events_cleaned.csv', index=False)
    print(f"✅ Saved full dataset: {len(df_clean):,} rows")
    
    # Also save a sample for quick testing
    sample = df_clean.sample(n=100000, random_state=42)
    sample.to_csv('../data/processed/events_sample.csv', index=False)
    print(f"✅ Saved sample dataset: {len(sample):,} rows")
else:
    df_clean.to_csv('../data/processed/events_cleaned.csv', index=False)
    print(f"✅ Saved cleaned dataset: {len(df_clean):,} rows")

# Save aggregated views
hourly.to_csv('../data/processed/hourly_aggregated.csv', index=False)
print(f"✅ Saved hourly aggregation: {len(hourly):,} rows")

daily.to_csv('../data/processed/daily_aggregated.csv', index=False)
print(f"✅ Saved daily aggregation: {len(daily):,} rows")

event_pivot.to_csv('../data/processed/events_by_type.csv', index=False)
print(f"✅ Saved event pivot table: {len(event_pivot):,} rows")

print("\n" + "=" * 50)
print("DATA PREPARATION COMPLETE!")
print("=" * 50)
print("\nReady for exploratory analysis!")

# %% [markdown]
# ### Summary Statistics

# %%
# Create summary statistics
summary_stats = {
    'Dataset': 'E-commerce Cosmetics Shop',
    'Time Period': f"{df_clean['event_time'].min().strftime('%Y-%m-%d')} to {df_clean['event_time'].max().strftime('%Y-%m-%d')}",
    'Total Events': f"{len(df_clean):,}",
    'Unique Users': f"{df_clean['user_id'].nunique():,}",
    'Unique Products': f"{df_clean['product_id'].nunique():,}",
    'Unique Sessions': f"{df_clean['user_session'].nunique():,}",
    'Unique Brands': f"{df_clean['brand'].nunique():,}",
    'View Events': f"{(df_clean['event_type'] == 'view').sum():,}",
    'Cart Events': f"{(df_clean['event_type'] == 'cart').sum():,}",
    'Purchase Events': f"{(df_clean['event_type'] == 'purchase').sum():,}",
    'Total Revenue': f"${df_clean[df_clean['event_type'] == 'purchase']['price'].sum():,.2f}"
}

summary_df = pd.DataFrame(list(summary_stats.items()), columns=['Metric', 'Value'])
print(summary_df.to_string(index=False))

# %%
print("\n✅ Notebook complete! Proceed to 02_exploratory_analysis.ipynb")

✅ Libraries imported successfully
Loading data...
✅ Data loaded successfully
Shape: (4102283, 9)

Columns: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

First few rows:
Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4102283 entries, 0 to 4102282
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int64  
 8   user_session   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 281.7+ MB


Basic Statistics:


Missing Values:
               Missing Count  Percentage
category_code        4034806   98.355135
brand                1659261   40.447258
user_session             637    0.015528
Unique Value Counts:
event_time: 1,782,439 unique